### Machine Learning Foundations Assignment

This notebook covers core Machine Learning concepts using 3 datasets.
* A user-defined Dataset [Student Performance]
* Kaggle Dataset [Titanic]
* Built-in Scikit-Learn Dataset [Iris]

I aimed to cover the following topics:
1. Descriptive Statistics
2. Probability
3. Probability Distributions
4. Sampling & CLT
5. Hypothesis Testing
6. Correlation & Covariance
7. Linear Algebra
8. Calculus Basics
9. Data Preprocessing (Scikit-Learn)
10. Regression
11. Classification
12. Clustering
13. PCA
14. Model Evaluation
15. Hyperparameter Tuning

In [ ]:
# importing all packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.tree import plot_tree
from scipy import stats

### Part 1: Student Performance Dataset [The Fundamentals]
We are starting with a small, custom dataset to look at how basic statistics and straight-line models work. Before running any complex algorithms, it helps to understand the spread of our data and how features relate to one another.

**Concepts Covered**:
* Descriptive Statistics [mean, median, and standard deviation]
* Correlation and Covariance
* Linear algebra and Calculus [weights and lines of best fit]
* Linear Regression

In [ ]:
data = {
    "student_id": range(1, 11),
    "hours_studied": [4.5, 7.0, 2.1, 9.5, 6.2, 1.5, 8.0, 5.5, 3.8, 8.5],
    "attendance": [85, 92, 70, 98, 88, 65, 95, 80, 75, 90],
    "assignments_completed": [8, 9, 5, 10, 8, 3, 9, 7, 6, 10],
    "sleep_hours": [7, 8, 6, 7, 5, 6, 8, 7, 6, 8],
    "test_score": [72, 88, 55, 96, 78, 48, 92, 75, 64, 94],
    "result": [1, 1, 0, 1, 1, 0, 1, 1, 0, 1]
}

dfstud = pd.DataFrame(data)

In [ ]:
print("First 5 records:")
dfstud.head()

In [ ]:
# describing dataset
dfstud.describe(include="all")

In [ ]:
# correlation analysis
corr_matrix = dfstud[["hours_studied", "attendance", "assignments_completed", "sleep_hours", "test_score"]].corr()
print(corr_matrix["test_score"])

In [ ]:
# visualisation
plt.figure(figsize=(14, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

Now I'll split the data to train a multiple linear regression model. The model calculates a weight coefficient for each feature to see how much it influences the final test score.

In [ ]:
# prediction and evaluation
x = dfstud[["hours_studied", "attendance", "assignments_completed", "sleep_hours"]]
y = dfstud["test_score"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

reg_model = LinearRegression()
reg_model.fit(x_train, y_train)

print("Intercept:", reg_model.intercept_)
print("Coefficients:", reg_model.coef_)

In [ ]:
# prediction and evaluation
y_pred = reg_model.predict(x_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Mean Absolute Error:", mae)
print("Root Mean Square Error:", rmse)

### Part 2: Titanic Dataset [The Real-World Mess]
Real-world data is rarely clean. In this section, we load the passenger data from the Titanic disaster to practice cleaning missing blocks, mapping text categories into numbers, and training tree-based classification models.

**Concepts Covered**:
* Data cleaning [filling missing cells, removing unhelpful columns]
* Categorical Encoding
* Grouping and Conditional Patterns
* Decision Trees
* Hyperparameter Optimisation

In [ ]:
path = "titanic.csv"
dftit = pd.read_csv(path)

In [ ]:
print("First 5 records:")
dftit.head()

In [ ]:
print("Dataset Info:")
dftit.info()

In [ ]:
print("Missing Values:")
dftit.isnull().sum()

In [ ]:
print(dftit["Survived"].value_counts())

In [ ]:
# handling missing data and dropping columns
median_age = dftit["Age"].median()
dftit["Age"] = dftit["Age"].fillna(median_age)
dftit = dftit.drop(columns=["Cabin"])
dftit = dftit.dropna(subset=["Embarked"])

print("Duplicates:", dftit.duplicated().sum())

In [ ]:
# converting categorical columns to numeric
dftit["Sex"] = dftit["Sex"].map({"male": 0, "female": 1})
dftit = pd.get_dummies(dftit, columns=["Embarked"], drop_first=True)
survival_by_gender = dftit.groupby("Sex")["Survived"].mean()
print(survival_by_gender)

In [ ]:
# visualisation
plt.figure(figsize=(14, 7))
sns.histplot(dftit["Age"], kde=True, color="teal")
plt.title("Passenger Age Distribution")
plt.show()

With our data cleaned, we can now train a decision tree to see how the model splits passengers into groups based on questions, and use grid search to find the optimal depth configuration.

In [ ]:
# training a decision tree classifier
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked_Q", "Embarked_S"]
xtit = dftit[features]
ytit = dftit["Survived"]

x_trainttit, x_testtit, y_traintit, y_testtit = train_test_split(xtit, ytit, test_size=0.2, random_state=42)

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(x_trainttit, y_traintit)

In [ ]:
# visualisation
plt.figure(figsize=(12, 6))
plot_tree(tree_model, feature_names=features, class_names=["Perished", "Survived"], filled=True)
plt.title("Decision Tree Splits")
plt.show()

In [ ]:
# hyperparameter tuning
param_grid = {"max_depth": [2, 3, 4, 5, 6]}
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid_search.fit(x_trainttit, y_traintit)

print("Best depth:", grid_search.best_params_["max_depth"])

In [ ]:
best_model = grid_search.best_estimator_
pred_tit = best_model.predict(x_testtit)
print("Classification Report:")
print(classification_report(y_testtit, pred_tit))

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_testtit, pred_tit))

### Part 3: Iris Dataset [The Unsupervised Patterns]
In this final section, we look at unsupervised techniques where the data points have no preset target labels. Using flower measurements, we want to see if mathematical clustering and dimension shrinking tools can discover natural patterns on their own.

**Concepts Covered**:
* Statistical Hypothesis Testing
* K-Means Clustering
* Principal Component Analysis [PCA] for lower-dimensional mapping

In [ ]:
# loading dataset
iris = load_iris()
dfiris = pd.DataFrame(data=iris.data, columns=iris.feature_names)
dfiris["species_id"] = iris.target

In [ ]:
print("First 5 records:")
print(dfiris.head())

In [ ]:
print("Iris Dataset Description:")
print(dfiris.describe())

In [ ]:
# independent t-test between species 0 and 1
setosa_sepal = dfiris[dfiris["species_id"] == 0]["sepal length (cm)"]
versicolor_sepal = dfiris[dfiris["species_id"] == 1]["sepal length (cm)"]

t_stat, p_val = stats.ttest_ind(setosa_sepal, versicolor_sepal)
print("P-value:", p_val)

Now we strip away the labels and use K-Means to discover clusters blindly. We finish by running PCA to compress our measurements down to two dimensions for a final visual validation.

In [ ]:
x_clustering = dfiris.drop(columns=["species_id"])

In [ ]:
# calculating variations for different k values
wcss = []
for k in range(1, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(x_clustering)
    wcss.append(kmeans.inertia_)

In [ ]:
# visualisation
plt.figure(figsize=(6, 4))
plt.plot(range(1, 8), wcss, marker="o", color="indigo")
plt.title("Elbow Method Curve")
plt.show()

In [ ]:
# kmeans clustering and pivot table
final_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
dfiris["cluster_id"] = final_kmeans.fit_predict(x_clustering)

pivot = dfiris.pivot_table(index="species_id", columns="cluster_id", aggfunc="size", fill_value=0)
print(pivot)

In [ ]:
# reducing features to two components
pca = PCA(n_components=2)
iris_pca = pca.fit_transform(x_clustering)

df_pca = pd.DataFrame(data=iris_pca, columns=["Principal Component 1", "Principal Component 2"])
df_pca["cluster_id"] = dfiris["cluster_id"]

print("Variance ratio:", pca.explained_variance_ratio_)

In [ ]:
# visualisation
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df_pca, x="Principal Component 1", y="Principal Component 2", hue="cluster_id", palette="viridis")
plt.title("PCA Cluster Representation")
plt.show()